In [ ]:
!pip install -q -U transformers sentencepiece sacremoses datasets accelerate evaluate bert-score nltk pandas plotly

In [1]:
import os
import time
import json
from pathlib import Path

import pandas as pd
import torch
from datasets import Dataset
from nltk.translate.bleu_score import corpus_bleu
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SRC_LANG = 'san_Deva'
TGT_LANG = 'eng_Latn'
MODEL_NAME = 'facebook/nllb-200-distilled-600M'
DATA_DIR = Path('.')
OUTPUT_DIR = Path('./artifacts')
CHECKPOINT_DIR = OUTPUT_DIR / 'best_checkpoint'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_LEN = 128
MAX_NEW_TOKENS = 128
NUM_BEAMS = 5
print({'device': DEVICE, 'model': MODEL_NAME})

C:\Users\rikin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'device': 'cpu', 'model': 'facebook/nllb-200-distilled-600M'}


In [ ]:
def load_split(split: str, count: str) -> pd.DataFrame:
    sa = pd.read_csv(DATA_DIR / f'{split}_sa_{count}.csv')
    path_en = DATA_DIR / f'{split}_en_{count}.csv'
    if path_en.exists():
        en = pd.read_csv(path_en)
        df = sa.merge(en, on='Source_id', how='inner')
    else:
        df = sa.copy()
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].fillna('').astype(str).str.strip()
    return df

train_df = load_split('train', '10000')
dev_df = load_split('dev', '1000')
test_df = load_split('test', '1000')
print('train/dev/test:', train_df.shape, dev_df.shape, test_df.shape)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, src_lang=SRC_LANG)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
forced_bos_id = tokenizer.convert_tokens_to_ids(TGT_LANG)
model.generation_config.forced_bos_token_id = forced_bos_id
#model.generation_config.max_new_tokens = max_new_tokens
model.generation_config.length_penalty = 1.0
model.generation_config.no_repeat_ngram_size = 3
model.generation_config.repetition_penalty = 1.2

In [ ]:
train_ds = Dataset.from_pandas(
    train_df[['Sentence_sa', 'Sentence_en']].rename(columns={'Sentence_sa': 'input_text', 'Sentence_en': 'target_text'}),
    preserve_index=False,
)
dev_ds = Dataset.from_pandas(
    dev_df[['Sentence_sa', 'Sentence_en']].rename(columns={'Sentence_sa': 'input_text', 'Sentence_en': 'target_text'}),
    preserve_index=False,
)

def preprocess(batch):
    model_inputs = tokenizer(batch['input_text'], max_length=MAX_LEN, truncation=True)
    labels = tokenizer(text_target=batch['target_text'], max_length=MAX_LEN, truncation=True)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
dev_tok = dev_ds.map(preprocess, batched=True, remove_columns=dev_ds.column_names)

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR / 'trainer_runs'),
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.05,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_strategy='steps',
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    fp16=(DEVICE == 'cuda'),
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=dev_tok,
    data_collator=data_collator,
)

train_result = trainer.train()
trainer.save_model(str(CHECKPOINT_DIR))
tokenizer.save_pretrained(str(CHECKPOINT_DIR))
pd.DataFrame(trainer.state.log_history).to_csv(OUTPUT_DIR / 'training_log.csv', index=False)
print(train_result.metrics)

In [2]:
best_model = AutoModelForSeq2SeqLM.from_pretrained(CHECKPOINT_DIR).to(DEVICE)
best_tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_DIR, src_lang=SRC_LANG)
forced_bos_id = best_tokenizer.convert_tokens_to_ids(TGT_LANG)

@torch.no_grad()
def translate_sentences(sentences, batch_size=16, num_beams=NUM_BEAMS, max_new_tokens=MAX_NEW_TOKENS):
    best_model.eval()
    preds = []
    for i in range(0, len(sentences), batch_size):
        batch = [str(x).strip() for x in sentences[i:i+batch_size]]
        enc = best_tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=MAX_LEN).to(DEVICE)
        gen = best_model.generate(
            **enc,
            forced_bos_token_id=forced_bos_id,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            early_stopping=True,
        )
        preds.extend(best_tokenizer.batch_decode(gen, skip_special_tokens=True))
    return preds



Loading weights: 100%|██████████| 509/509 [00:00<00:00, 4377.77it/s]


In [ ]:
dev_preds = translate_sentences(dev_df['Sentence_sa'].tolist())
test_preds = translate_sentences(test_df['Sentence_sa'].tolist())

submission = pd.DataFrame({'Source_id': test_df['Source_id'], 'Sentence_en': test_preds})
submission.to_csv(OUTPUT_DIR / 'submission.csv', index=False, encoding='utf-8')

references = [[ref.split()] for ref in dev_df['Sentence_en'].tolist()]
hypotheses = [pred.split() for pred in dev_preds]
bleu = corpus_bleu(references, hypotheses)

metrics = {
    'dev_bleu': float(bleu),
    'parameter_count': int(sum(p.numel() for p in best_model.parameters())),
    'checkpoint_dir': str(CHECKPOINT_DIR),
}
with open(OUTPUT_DIR / 'metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

pd.DataFrame({
    'Source_id': test_df['Source_id'],
    'Sentence_sa': test_df['Sentence_sa'],
    'Prediction_en': test_preds,
}).to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)

In [3]:
from bert_score import score as bertscore_fn
def run_on_new_test(new_test_sa_csv, new_test_en_csv=None, out_csv='new_submission.csv'):
    new_sa = pd.read_csv(new_test_sa_csv)
    if new_test_en_csv and Path(new_test_en_csv).exists():
        new_en = pd.read_csv(new_test_en_csv)
        new_df = new_sa.merge(new_en, on='Source_id', how='inner')
    else:
        new_df = new_sa.copy()

    for col in new_df.columns:
        if new_df[col].dtype == 'object':
            new_df[col] = new_df[col].fillna('').astype(str).str.strip()

    test_start = time.time()
    preds = translate_sentences(new_df['Sentence_sa'].tolist())
    test_inference_time = time.time() - test_start

    out_df = pd.DataFrame({'Source_id': new_df['Source_id'], 'Sentence_en': preds})
    out_df.to_csv(OUTPUT_DIR / out_csv, index=False, encoding='utf-8')

    if 'Sentence_en' in new_df.columns:
        refs = [[ref.split()] for ref in new_df['Sentence_en'].tolist()]
        hyps = [pred.split() for pred in preds]
        print('BLEU:', corpus_bleu(refs, hyps))
        P, R, F1 = bertscore_fn(preds, new_df['Sentence_en'].tolist(), lang='en', rescale_with_baseline=True)
        print('BERTScore:', P.mean(), R.mean(), F1.mean())
        print('Inference time:', test_inference_time)
    total_params = sum(p.numel() for p in best_model.parameters())
    print(f"Total parameters: {total_params:,}")

        



    return out_df

In [4]:
run_on_new_test('./dataset/test_demo_sa.csv', './dataset/test_demo_en.csv', 'my_new_test_submission.csv')

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

BLEU: 0.21113035550651693


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 4293.09it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
C:\Users\rikin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\bert_score\score.py:149: UserWarning: The given NumPy array is not writable, and PyTorch does not support 

BERTScore: tensor(0.6023) tensor(0.5558) tensor(0.5791)
Inference time: 264.8478994369507
Total parameters: 615,073,792


,Source_id,Sentence_en
0,1001,"""For I would not, brethren, that ye should be ..."
1,1002,Benefits: Strengthening and smoothness of the ...
2,1003,"In this tutorial, we learnt: how to assign an ..."
3,1004,"When we are on this course page, the menu on t..."
4,1005,The names of the students who wish to particip...
...,...,...
95,1096,It is compiled as follows:
96,1097,The water also reaches the topmost leaves of t...
97,1098,"So he went straight to the commander and said,..."
98,1099,"First, double the layer. For example, check wh..."


# Without Reference

In [ ]:
run_on_new_test('./test_sa_1000.csv', None, 'my_new_unlabeled_submission.csv')